In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType

CATALOG_SCHEMA = "workspace.ecommerce_de"

# 1. Orders Data Quality & Quarantine
orders_bronze = spark.table(f"{CATALOG_SCHEMA}.bronze_orders")

orders_checked = (
    orders_bronze.withColumn("amt_num", F.col("order_amount").cast(DoubleType()))
    .withColumn(
        "rejection_reason",
        F.concat_ws(
            " | ",
            F.when(
                F.col("order_id").isNull() | (F.trim(F.col("order_id")) == ""),
                F.lit("NULL_ORDER_ID"),
            ),
            F.when(
                F.col("customer_id").isNull() | (F.trim(F.col("customer_id")) == ""),
                F.lit("NULL_CUSTOMER_ID"),
            ),
            F.when(F.col("amt_num").isNull(), F.lit("INVALID_OR_NULL_AMOUNT")),
            F.when(F.col("amt_num") <= 0, F.lit("NON_POSITIVE_AMOUNT")),
            F.when(
                ~F.col("status").isin("COMPLETED", "PENDING", "CANCELLED", "REFUNDED"),
                F.lit("INVALID_STATUS"),
            ),
        ),
    )
    .withColumn(
        "rejection_reason",
        F.when(F.col("rejection_reason") == "", F.lit(None)).otherwise(F.col("rejection_reason")),
    )
)

quarantine_orders_df = (
    orders_checked.filter(F.col("rejection_reason").isNotNull())
    .drop("amt_num")
    .withColumn("quarantined_at", F.current_timestamp())
)

valid_orders_df = (
    orders_checked.filter(F.col("rejection_reason").isNull())
    .drop("rejection_reason")
    .withColumn("order_amount_clean", F.col("amt_num"))
    .drop("amt_num")
)

quarantine_orders_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG_SCHEMA}.quarantine_orders"
)
valid_orders_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG_SCHEMA}.staging_valid_orders"
)

# 2. Payments Data Quality & Quarantine ({amount, method, status} format)
payments_bronze = spark.table(f"{CATALOG_SCHEMA}.bronze_payments")

cleaned_pay_expr = F.regexp_replace(F.trim(F.col("payment")), r"^\{|\}$", "")
pay_parts = F.split(cleaned_pay_expr, ",")

payments_checked = (
    payments_bronze.withColumn(
        "pay_amt_num",
        F.trim(F.element_at(pay_parts, 1)).cast(DoubleType()),
    )
    .withColumn(
        "rejection_reason",
        F.when(
            F.col("event_id").isNull() | (F.trim(F.col("event_id")) == ""),
            "NULL_EVENT_ID",
        )
        .when(
            F.col("order_id").isNull() | (F.trim(F.col("order_id")) == ""),
            "NULL_ORDER_ID",
        )
        .when(
            F.col("pay_amt_num").isNull() | (F.col("pay_amt_num") < 0),
            "INVALID_PAYMENT_AMOUNT",
        ),
    )
)

quarantine_payments_df = (
    payments_checked.filter(F.col("rejection_reason").isNotNull())
    .drop("pay_amt_num")
    .withColumn("quarantined_at", F.current_timestamp())
)

valid_payments_df = (
    payments_checked.filter(F.col("rejection_reason").isNull())
    .drop("rejection_reason", "pay_amt_num")
)

quarantine_payments_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG_SCHEMA}.quarantine_payments"
)
valid_payments_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(
    f"{CATALOG_SCHEMA}.staging_valid_payments"
)

print("✅ Day 3 complete: Quarantine & Valid staging tables updated.")

In [0]:
%sql
SELECT 'staging_valid_orders' AS tbl, COUNT(*) AS cnt FROM workspace.ecommerce_de.staging_valid_orders
UNION ALL
SELECT 'quarantine_orders', COUNT(*) FROM workspace.ecommerce_de.quarantine_orders
UNION ALL
SELECT 'staging_valid_payments', COUNT(*) FROM workspace.ecommerce_de.staging_valid_payments
UNION ALL
SELECT 'quarantine_payments', COUNT(*) FROM workspace.ecommerce_de.quarantine_payments;